# Support Vector Machines (SVM)

An SVM finds the **best boundary** between two groups of points. In this lab we build up the main ideas visually, one picture at a time:

1. An SVM finds the **widest margin** between two groups
2. The `C` setting: strict vs. relaxed
3. The **kernel trick**: separating shapes a straight line can't
4. Different **kernels** and their effect

Run each cell with **Shift + Enter** and look at the plot it produces.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # enables 3D plots
from sklearn.svm import SVC
from sklearn.datasets import make_blobs, make_circles

## The plotting helper

We use a single helper, `plot_svm`, to visualize what each model learns:

- the background is shaded into the two regions the SVM predicts
- the dots are the data, colored by their true class
- circled dots are the **support vectors** — the points that determine the boundary

In [ ]:
def plot_svm(model, X, y, title=""):
    plt.figure(figsize=(5, 5))
    xx, yy = np.meshgrid(np.linspace(X[:,0].min()-1, X[:,0].max()+1, 300),
                         np.linspace(X[:,1].min()-1, X[:,1].max()+1, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
    plt.scatter(X[:,0], X[:,1], c=y, cmap="coolwarm", edgecolors="k")
    sv = model.support_vectors_
    plt.scatter(sv[:,0], sv[:,1], s=130, facecolors="none", edgecolors="k")
    plt.title(title); plt.xticks([]); plt.yticks([]); plt.show()

## 1) The widest margin

An SVM separates two groups with the boundary that leaves the **widest margin** — the most empty space on either side. The points touching the edge of the margin are the **support vectors**.

In [ ]:
X, y = make_blobs(n_samples=80, centers=2, cluster_std=1.2, random_state=6)

In [ ]:
model = SVC(kernel="linear")
model.fit(X, y)
plot_svm(model, X, y, "Linear SVM (maximum margin)")

The circled points are the support vectors. The boundary sits in the middle of the widest gap between the two groups.

## 2) Soft margin: the `C` setting

When groups overlap, `C` controls the trade-off between a wide margin and classifying every point correctly:

- **small C** — a wider margin that tolerates some points on the wrong side
- **large C** — a narrower margin that tries to classify every point correctly

In [ ]:
X, y = make_blobs(n_samples=80, centers=2, cluster_std=2.0, random_state=6)

In [ ]:
model = SVC(kernel="linear", C=0.05).fit(X, y)
plot_svm(model, X, y, "Small C (wide margin)")

In [ ]:
model = SVC(kernel="linear", C=100).fit(X, y)
plot_svm(model, X, y, "Large C (narrow margin)")

Smaller `C` gives a simpler, wider boundary; larger `C` bends to fit individual points. A wider margin often generalizes better to unseen data.

## 3) The kernel trick

Here one group is a small circle and the other is a ring around it. No straight line can separate them.

In [ ]:
X, y = make_circles(n_samples=200, factor=0.4, noise=0.08, random_state=0)

In [ ]:
model = SVC(kernel="linear").fit(X, y)
plot_svm(model, X, y, "Linear kernel on ring data")

A straight line cannot separate the rings.

The idea behind the kernel trick: add a new feature equal to each point's distance from the center (`height = x² + y²`). Inner points stay low, the outer ring lifts up, and a flat plane can then separate them.

In [ ]:
z = X[:,0]**2 + X[:,1]**2
ax = plt.figure().add_subplot(projection="3d")
ax.scatter(X[:,0], X[:,1], z, c=y, cmap="coolwarm", edgecolors="k")
ax.set_title("Lifted to 3D: a plane now separates the classes")
plt.show()

The **RBF kernel** performs this lifting implicitly, without us choosing the feature. It finds a boundary that wraps around the inner circle:

In [ ]:
model = SVC(kernel="rbf").fit(X, y)
plot_svm(model, X, y, "RBF kernel on ring data")

The RBF kernel produces a circular boundary, which a linear kernel cannot.

## 4) Different kernels

A kernel determines the shape of boundary the SVM can draw. We compare a few on the same ring data.

In [ ]:
model = SVC(kernel="poly", degree=2).fit(X, y)
plot_svm(model, X, y, "Polynomial kernel (degree 2)")

In [ ]:
model = SVC(kernel="rbf", gamma=0.5).fit(X, y)
plot_svm(model, X, y, "RBF kernel, gamma=0.5")

Both the polynomial and RBF kernels can draw curved boundaries, so they separate the rings that a linear kernel cannot. The RBF kernel's `gamma` setting controls how closely the boundary follows the data.

## Recap

- **SVM** — the boundary with the **widest margin** between two groups.
- **Support vectors** — the points that determine the boundary.
- **C** — strict (large) vs. relaxed (small) margin.
- **Kernel trick** — add features (lift to a higher dimension) so a flat separator works.
- **RBF / polynomial** kernels produce curved boundaries; **gamma** controls flexibility.

## Exercises

Change a value, re-run the cell, and observe the result:
- In the ring data, set `noise=0.15` — does the RBF kernel still separate the classes?
- Set `gamma=50` — is the boundary still reasonable, or does it overfit?
- In step 2, increase `C` to `1000` — how narrow does the margin become?
- Change `degree=2` to `degree=3` in the polynomial kernel.

---
*Contributed by: Sattam Altwaim*